In [ ]:
!pip uninstall -y pyspark delta-spark dataproc-spark-connect
!pip install pyspark==3.5.0 delta-spark==3.2.0

Found existing installation: pyspark 3.5.0
Uninstalling pyspark-3.5.0:
  Successfully uninstalled pyspark-3.5.0
Found existing installation: delta-spark 3.2.0
Uninstalling delta-spark-3.2.0:
  Successfully uninstalled delta-spark-3.2.0
  Using cached pyspark-3.5.0-py2.py3-none-any.whl
  Using cached delta_spark-3.2.0-py3-none-any.whl.metadata (2.0 kB)
Using cached delta_spark-3.2.0-py3-none-any.whl (21 kB)


In [ ]:
import sys
print(sys.version)

import pyspark
print("PySpark:", pyspark.__version__)
print("Path:", pyspark.__file__)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PySpark: 3.5.0
Path: /usr/local/lib/python3.12/dist-packages/pyspark/__init__.py


In [ ]:
!pip show delta-spark

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 46, in run
    if not print_results(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 178, in print_results
    for i, dist in enumerate(distributions):
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 114, in search_packages_info
    required_by = sorted(_get_requiring_packages(dist), key=str.lower)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 95, in <genexpr>
    in {canonicalize_name(d.name) for d in dist.iter_dependencies()}
               

In [ ]:
import platform
print(platform.platform())

Linux-6.6.122+-x86_64-with-glibc2.35


In [ ]:
!pip uninstall -y delta-spark
!pip install delta-spark==3.2.0

Found existing installation: delta-spark 3.2.0
Uninstalling delta-spark-3.2.0:
  Successfully uninstalled delta-spark-3.2.0
  Using cached delta_spark-3.2.0-py3-none-any.whl.metadata (2.0 kB)
Using cached delta_spark-3.2.0-py3-none-any.whl (21 kB)


In [ ]:
from pyspark.sql.column import _to_seq

In [ ]:
import json, os
from collections import defaultdict

class MiniKafka:
    def __init__(self, base_path="/content/mini_kafka"):
        self.base_path = base_path
        os.makedirs(base_path, exist_ok=True)

    def produce(self, topic, message: dict):
        path = f"{self.base_path}/{topic}.jsonl"
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(message, ensure_ascii=False) + "\n")

    def consume(self, topic):
        path = f"{self.base_path}/{topic}.jsonl"
        if not os.path.exists(path):
            return []
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f]

    def create_topic(self, topic):
        open(f"{self.base_path}/{topic}.jsonl", "a").close()

mk = MiniKafka()
mk.create_topic("applicants")
mk.create_topic("applicants_dlq")

print("MiniKafka Ready")

MiniKafka Ready


In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder.appName("AI Recruitment Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark + Delta:", spark.version)

Spark + Delta: 3.5.0


In [ ]:
import importlib.metadata as md

print("PySpark version:", md.version("pyspark"))
print("Delta-Spark version:", md.version("delta-spark"))

PySpark version: 3.5.0
Delta-Spark version: 3.2.0


In [ ]:

from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks/AI_Recruitment"
DELTA = f"{BASE}/delta"

BRONZE_APP = f"{DELTA}/bronze_applicants"
BRONZE_JOB = f"{DELTA}/bronze_jobs"
SILVER     = f"{DELTA}/silver_applications"
GOLD_JOB   = f"{DELTA}/gold_job_demand"
GOLD_CITY  = f"{DELTA}/gold_city_stats"

valid_pd = pd.read_json(f"{BASE}/output/bronze_applicants.json")
jobs_pd  = pd.read_csv(f"{BASE}/data/jobs.csv")

print("Applied:", len(valid_pd), "| Jobs:", len(jobs_pd))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Applied: 32 | Jobs: 5


In [ ]:

from pyspark.sql.functions import concat_ws, col

valid_pd["skills"] = valid_pd["skills"].apply(
    lambda s: ", ".join(s) if isinstance(s, list) else s)

bronze_app = spark.createDataFrame(valid_pd)
bronze_job = spark.createDataFrame(jobs_pd)

bronze_app.write.format("delta").mode("overwrite").save(BRONZE_APP)
bronze_job.write.format("delta").mode("overwrite").save(BRONZE_JOB)

print("Bronze (Delta) its been created")
bronze_app.show(5)



Bronze (Delta) its been created
+------------+------+----------------+---------+----------------+--------------------+
|applicant_id|job_id|            name|education|experience_years|              skills|
+------------+------+----------------+---------+----------------+--------------------+
|        1007|     2|Abdulrahman Saad| Bachelor|               2|         Python, SQL|
|        1006|     1|   Hind Abdullah| Bachelor|               1|           HTML, CSS|
|        1004|     4|     Noura Fahad| Bachelor|               3|Node.js, SQL, Lar...|
|        1008|     5|      Reem Saleh|   Master|               4|  TensorFlow, Python|
|        1005|     5|    Faisal Ahmed|   Master|               5|Python, TensorFlo...|
+------------+------+----------------+---------+----------------+--------------------+
only showing top 5 rows



In [ ]:

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

bad_schema = StructType([
    StructField("applicant_id", StringType(), True),
    StructField("job_id", IntegerType(), True),
    StructField("name", StringType(), True),
])
bad_df = spark.createDataFrame([("XYZ", 1, "Bad Record")], bad_schema)

try:
    bad_df.write.format("delta").mode("append").save(BRONZE_APP)
    print("Warning: Write operation performed — should have been rejected")
except Exception as e:
    print("Write rejected by Delta schema enforcement")
    print("Reason:", str(e)[:400])


Write rejected by Delta schema enforcement
Reason: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'applicant_id' and 'applicant_id'


In [ ]:
from pyspark.sql.functions import col, concat_ws

silver = (
    bronze_app.alias("a")
    .join(bronze_job.alias("j"), "job_id")
    .select(
        col("job_id"),
        col("a.applicant_id"),
        col("a.name"),
        col("a.education"),
        col("a.experience_years").alias("applicant_experience"),
        col("a.skills"),
        col("j.title"),
        col("j.company"),
        col("j.city"),
        col("j.salary"),
        col("j.required_skills"),
        col("j.experience_years").alias("required_experience")
    )
    .withColumn(
        "application_id",
        concat_ws("_", col("applicant_id"), col("job_id"))
    )
)

silver.write.format("delta").mode("overwrite").save(SILVER)

In [ ]:
print(silver.columns)

In [ ]:
import pandas as pd

existing = [r["applicant_id"] for r in
            silver.select("applicant_id").limit(2).collect()]

updates_pd = pd.DataFrame([
    {
        "applicant_id": existing[0],
        "job_id": 2,
        "name": "Updated A",
        "education": "Master",
        "experience_years": 9,
        "skills": "Python, Spark, Kafka"
    },
    {
        "applicant_id": existing[1],
        "job_id": 1,
        "name": "Updated B",
        "education": "PhD",
        "experience_years": 11,
        "skills": "Vue.js, JavaScript"
    },
    {
        "applicant_id": 999999,
        "job_id": 3,
        "name": "Brand New",
        "education": "Bachelor",
        "experience_years": 1,
        "skills": "Figma, UX"
    }
])

In [19]:
from delta.tables import DeltaTable

silver_tbl = DeltaTable.forPath(spark, SILVER)

before = spark.read.format("delta").load(SILVER).count()
print("Before MERGE:", before)

(silver_tbl.alias("t")
 .merge(
     updates.alias("s"),
     "t.applicant_id = s.applicant_id"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

after = spark.read.format("delta").load(SILVER).count()
print("After MERGE:", after)
print("Inserted:", after - before)

Before MERGE: 32
After MERGE: 33
Inserted: 1


In [20]:
merged = spark.read.format("delta").load(SILVER)

print(" Updated Records:")
merged.filter(col("applicant_id").isin(existing)) \
      .select(
          "applicant_id",
          "name",
          "education",
          "applicant_experience",
          "required_experience"
      ).show(truncate=False)

print(" The New Record:")
merged.filter(col("applicant_id") == 999999) \
      .select(
          "applicant_id",
          "name",
          "title"
      ).show(truncate=False)

print(" Table History (Delta):")
silver_tbl.history() \
          .select("version", "operation") \
          .show(5, truncate=False)

 Updated Records:
+------------+---------+---------+--------------------+-------------------+
|applicant_id|name     |education|applicant_experience|required_experience|
+------------+---------+---------+--------------------+-------------------+
|1001        |Updated B|PhD      |11                  |2                  |
|1001        |Updated B|PhD      |11                  |2                  |
|1001        |Updated B|PhD      |11                  |2                  |
|1001        |Updated B|PhD      |11                  |2                  |
|1006        |Updated A|Master   |9                   |4                  |
|1006        |Updated A|Master   |9                   |4                  |
|1006        |Updated A|Master   |9                   |4                  |
|1006        |Updated A|Master   |9                   |4                  |
+------------+---------+---------+--------------------+-------------------+

 The New Record:
+------------+---------+-----------+
|applicant_id|n